# Pretrained topic-path embeddings in 2D with t-SNE

Each point represents one of the 1,000 paths in `top_1000_topics.csv`. The projection is fitted on those paths. Its position comes from the L2-normalized pretrained topic-path embedding. Color identifies the main topic and shape identifies the subtopic within it. Hover for the full path and topic depth, or click a main topic in the legend to toggle its points.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "topic_embeddings").is_dir())
DATA_FILE = ROOT / "data/processed/topic_path_embeddings.csv"
SELECTED_TOPICS = ROOT / "data/processed/top_1000_topics.csv"
TOPIC_REFERENCE = ROOT / "results/sgns/topic_embeddings_normalized.csv"
PLOT_DIR = ROOT / "output"
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]
PERPLEXITY = 20
RANDOM_STATE = 42

## Load and project the path embeddings

In [12]:
selected = pd.read_csv(SELECTED_TOPICS, usecols=TOPIC_COLUMNS, dtype=str, keep_default_na=False)
if len(selected) != 1000:
    raise ValueError(f"Expected 1,000 selected topic paths, found {len(selected):,}")

embeddings = pd.read_csv(DATA_FILE, keep_default_na=False)
means = selected.merge(embeddings, on=TOPIC_COLUMNS, how="left", validate="one_to_one", indicator=True)
missing = means.loc[means["_merge"] == "left_only", TOPIC_COLUMNS]
if not missing.empty:
    raise ValueError(f"Missing label embeddings for {len(missing)} selected paths:\n{missing.head()}")
means = means.drop(columns="_merge")
means["Topic_depth"] = means[TOPIC_COLUMNS].ne("").sum(axis=1)
dimensions = [c for c in means if c.startswith("dim_")]
vectors = normalize(means[dimensions].to_numpy(dtype=np.float32))
projection_df = means[[*TOPIC_COLUMNS, "Topic_depth"]].copy()
projection_df["Topic_path"] = means[TOPIC_COLUMNS].apply(
    lambda row: " / ".join(label for label in row if label), axis=1
)
tsne = TSNE(
    n_components=2, perplexity=PERPLEXITY, init="pca", learning_rate="auto",
    metric="euclidean", max_iter=1_000, random_state=RANDOM_STATE,
)
if len(vectors) <= PERPLEXITY:
    raise ValueError("PERPLEXITY must be smaller than the number of topics")
projection_df[["TSNE_1", "TSNE_2"]] = tsne.fit_transform(vectors)
print(f"Projected {len(means):,} topic paths from {len(dimensions)} dimensions.")

Projected 1,000 topic paths from 896 dimensions.


## Plot by main topic and subtopic

When the topic embedding export is available, reuse its main-topic color order and alphabetical subtopic shape assignments. Assign any additional labels after the reference labels.

In [13]:
marker_symbols = [
    "circle", "square", "diamond", "cross", "x",
    "triangle-up", "triangle-down", "triangle-left", "triangle-right",
    "pentagon", "hexagon", "star", "hexagram", "star-triangle-up",
    "star-triangle-down", "star-square", "star-diamond", "diamond-tall",
    "diamond-wide", "hourglass", "bowtie", "circle-cross", "circle-x",
    "square-cross", "square-x", "diamond-cross", "diamond-x",
]
marker_symbols += [symbol + "-open" for symbol in marker_symbols]
reference = means
if TOPIC_REFERENCE.exists():
    reference = pd.read_csv(TOPIC_REFERENCE, usecols=TOPIC_COLUMNS[:2], keep_default_na=False)

main_topics = pd.concat([reference["Topic_1"], means["Topic_1"]]).unique().tolist()
topic2_symbols = {}
for topic1, group in means.groupby("Topic_1"):
    known = sorted(reference.loc[reference["Topic_1"] == topic1, "Topic_2"].unique())
    labels = known + sorted(set(group["Topic_2"]) - set(known))
    if len(labels) > len(marker_symbols):
        raise ValueError(f"Add more marker symbols for {topic1}")
    topic2_symbols[topic1] = dict(zip(labels, marker_symbols))

In [14]:
fig = px.scatter(
    projection_df,
    x="TSNE_1", y="TSNE_2", color="Topic_1", symbol="Topic_2",
    category_orders={"Topic_1": main_topics},
    custom_data=["Topic_1", "Topic_2"],
    render_mode="svg",
    hover_name="Topic_path",
    hover_data={"Topic_depth": ":,", "Topic_1": False, "Topic_2": False, "TSNE_1": False, "TSNE_2": False},
    title=f"Top 1,000 topic-label embeddings · t-SNE (perplexity={PERPLEXITY}, seed={RANDOM_STATE})",
    labels={"Topic_1": "Main topic", "Topic_depth": "Topic depth", "Topic_2": "Subtopic", "TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"},
    template="plotly_white",
    height=800,
)
# Each trace contains one (Topic_1, Topic_2) pair.
# Show one legend entry per main topic; clicking it toggles all its subtopics.
seen_topic1 = set()
for trace in fig.data:
    topic1, topic2 = trace.customdata[0][:2]
    trace.marker.symbol = topic2_symbols[topic1][topic2]
    trace.name = topic1
    trace.legendgroup = topic1
    trace.showlegend = topic1 not in seen_topic1
    seen_topic1.add(topic1)
fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.update_layout(legend_title_text="Main topic", legend_groupclick="togglegroup")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(PLOT_DIR / "topic_path_embeddings_tsne_shapes.html", include_plotlyjs=True)
fig.show()

# Examine neighbors in original embedding space

In [16]:
query_mask = means["Topic_2"].eq("wikileaks")
if not query_mask.any():
    raise ValueError("Topic_2='wikileaks' was not found")

query_indices = np.flatnonzero(query_mask.to_numpy())
query_vector = vectors[query_indices].mean(axis=0)
similarities = vectors @ query_vector

# Exclude the Wikileaks row(s) themselves
similarities[query_indices] = -np.inf
neighbor_indices = np.argsort(similarities)[::-1][:10]

neighbors = means.loc[neighbor_indices, TOPIC_COLUMNS + ["Topic_path"]].copy()
neighbors.insert(0, "cosine_similarity", similarities[neighbor_indices])
neighbors.reset_index(drop=True)

,cosine_similarity,Topic_1,Topic_2,Topic_3,Topic_path
0,0.647275,international,uno,,international / uno
1,0.642800,international,diplomatie,,international / diplomatie
2,0.635893,international,abhoerskandal,,international / abhoerskandal
3,0.635435,international,europaszukunft,,international / europaszukunft
4,0.628728,web,netzpolitik,wikileaks2010,web / netzpolitik / wikileaks2010
5,0.611623,international,nato,,international / nato
6,0.610024,international,jihadismus,,international / jihadismus
7,0.592674,international,prantnersperspektiven,,international / prantnersperspektiven
8,0.591792,international,ub-eastblog,,international / ub-eastblog
9,0.574663,international,flucht-und-politik,,international / flucht-und-politik
